# Gemma Guide Hackathon Demo Bootstrap

This notebook is the end-to-end bootstrap path for a fresh runtime.

It takes you from zero to a phone-openable HTTPS link by doing the following:

1. clone the repo
2. install dependencies
3. log into Hugging Face if needed
4. start the Gemma vLLM server
5. start `app_blind.py` in notebook/tunnel mode
6. expose the app through a Cloudflare Quick Tunnel
7. print an `https://...trycloudflare.com` link for your phone

The notebook is written to be usable on Kaggle, Colab, or a Linux GPU notebook VM.

Default demo profile locked in here:
- Gemma: `google/gemma-4-E4B-it`
- quantization: `bitsandbytes`
- served model name: `gemma-4-e4b-it`
- TIPS short-side: `672`
- target: fit and run end-to-end under a T4-class 16 GB budget

## Before you run it

- attach a GPU runtime
- make sure outbound internet is allowed
- accept the Hugging Face licenses for any Gemma / TIPS models you plan to use
- if you want a different model profile, change the config cell below; otherwise use the defaults as-is

## Runtime Notes

- The app itself runs on plain local HTTP inside the notebook VM.
- `cloudflared` provides the public HTTPS front door.
- Phone microphone access depends on opening the final `https://...trycloudflare.com` URL directly in the phone browser.
- If the runtime resets, rerun the notebook from the top.

## Token Setup

Paste your tokens into the cell below before running the clone and Hugging Face login steps.

- `GITHUB_TOKEN` is needed if the repo is private.
- `HF_TOKEN` is needed if model download requires authenticated Hugging Face access.

These values only live inside the current notebook runtime unless you export them somewhere else.

In [ ]:
import os

# Paste tokens here if they are not already set in the runtime environment.
GITHUB_TOKEN = ""
HF_TOKEN = ""

if GITHUB_TOKEN:
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

print('GITHUB_TOKEN set:', bool(os.getenv('GITHUB_TOKEN')))
print('HF_TOKEN set:', bool(os.getenv('HF_TOKEN')))

GITHUB_TOKEN set: True
HF_TOKEN set: True


In [2]:
#!rm -rf /content/Gemma4-VisionAssistant

In [3]:
from pathlib import Path
import os
import platform

REPO_URL = "https://github.com/pariidanDKE/Gemma4-VisionAssistant.git"
REPO_REF = "demo-setup"

if Path('/kaggle/working').exists():
    REPO_DIR = Path('/kaggle/working/Gemma4-VisionAssistant')
elif Path('/content').exists():
    REPO_DIR = Path('/content/Gemma4-VisionAssistant')
else:
    REPO_DIR = Path.cwd() / 'Gemma4-VisionAssistant'

HF_TOKEN = os.getenv('HF_TOKEN', '')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')

if GITHUB_TOKEN and REPO_URL.startswith('https://github.com/'):
    CLONE_URL = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
else:
    CLONE_URL = REPO_URL

# Demo profile
VLLM_MODEL_REPO  = "google/gemma-4-E4B-it"
VLLM_SERVED_NAME = "gemma-4-e4b-it"
VLLM_QUANTIZATION = "bitsandbytes"

# T4 memory profile
GPU_MEM_UTIL   = "0.75"
MAX_MODEL_LEN  = "2048"   # halves KV cache vs 4096
MAX_NUM_SEQS   = "1"
MAX_SOFT_TOKENS = "140"   # halves VRAM per image vs 280

# Extra vLLM flags (space-separated) — appended to the serve command
VLLM_EXTRA_ARGS = "--enforce-eager --max-num-batched-tokens 512"

TIPS_SHORT_SIDE = "672"
APP_HOST = "127.0.0.1"
APP_PORT = "7862"

print(f"Python: {platform.python_version()}")
print(f"Repo dir: {REPO_DIR}")
print(f"Model: {VLLM_MODEL_REPO}  served as: {VLLM_SERVED_NAME}")
print(f"gpu_mem_util={GPU_MEM_UTIL}  max_model_len={MAX_MODEL_LEN}  max_soft_tokens={MAX_SOFT_TOKENS}")
print(f"extra args: {VLLM_EXTRA_ARGS}")

Python: 3.12.13
Repo dir: /content/Gemma4-VisionAssistant
Model: google/gemma-4-E4B-it  served as: gemma-4-e4b-it
gpu_mem_util=0.75  max_model_len=2048  max_soft_tokens=140
extra args: --enforce-eager --max-num-batched-tokens 512


## Clone The Repo

If the repo already exists in the runtime, this cell leaves it in place.

In [4]:
import subprocess

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", CLONE_URL, str(REPO_DIR)], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], check=True)

CompletedProcess(args=['git', '-C', '/content/Gemma4-VisionAssistant', 'rev-parse', '--short', 'HEAD'], returncode=0)

## Install Dependencies

This installs the Python packages the current app path relies on, plus notebook-side helpers.

In [5]:
%pip install -q -U pip
%pip install -q vllm[audio]==0.20.1 transformers==5.7.0 gradio==6.14.0 pydub==0.25.1 gtts==2.5.4 openai==2.33.0 "bitsandbytes>=0.48.1"

In [6]:
import shutil
import subprocess

if shutil.which('ffmpeg') is None:
    print('ffmpeg not found; attempting install...')
    subprocess.run(['bash', '-lc', 'apt-get update -y && apt-get install -y ffmpeg'], check=True)
else:
    print('ffmpeg already available')

ffmpeg already available


## Hugging Face Login

Set `HF_TOKEN` in the runtime environment before running this cell, or paste it here directly if you are comfortable doing so in this notebook session.

In [7]:
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged into Hugging Face using HF_TOKEN from the environment.')
else:
    print('HF_TOKEN is empty. If model download fails, set HF_TOKEN and rerun this cell.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged into Hugging Face using HF_TOKEN from the environment.


## Start vLLM

This uses the repo's `scripts/start_gemma4.sh`, but drives it with explicit environment variables so the notebook is stable even if the repo defaults drift.

In [8]:
import subprocess, pathlib, os

# bitsandbytes>=0.45 needs libnvJitLink.so.13 (CUDA 12.3+).
# Colab T4 ships CUDA 12.2, which has the same lib under a different .so version number.
# We find whatever version exists and symlink it to the name bitsandbytes expects.

target = pathlib.Path('/usr/local/cuda/lib64/libnvJitLink.so.13')

if target.exists():
    print(f'libnvJitLink.so.13 already present at {target} — nothing to do.')
else:
    result = subprocess.run(
        ['find', '/usr/local', '/usr/lib', '-name', 'libnvJitLink.so.*'],
        capture_output=True, text=True
    )
    candidates = [p for p in result.stdout.strip().splitlines() if p]
    print('Found candidates:', candidates)

    if candidates:
        src = candidates[0]
        subprocess.run(['ln', '-sf', src, str(target)], check=True)
        print(f'Symlinked {src} → {target}')
    else:
        raise RuntimeError(
            'No libnvJitLink.so.* found. '
            'Try: apt-get install -y cuda-nvjitlink-12-3'
        )

# Ensure the CUDA lib dir is on LD_LIBRARY_PATH for child processes (vLLM)
cuda_lib = '/usr/local/cuda/lib64'
ld = os.environ.get('LD_LIBRARY_PATH', '')
if cuda_lib not in ld.split(':'):
    os.environ['LD_LIBRARY_PATH'] = cuda_lib + (':' + ld if ld else '')
    print(f'Added {cuda_lib} to LD_LIBRARY_PATH')

print('Done.')

libnvJitLink.so.13 already present at /usr/local/cuda/lib64/libnvJitLink.so.13 — nothing to do.
Added /usr/local/cuda/lib64 to LD_LIBRARY_PATH
Done.


In [9]:
import json
import os
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

LOG_DIR = Path('/tmp/gemma_demo_logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)
VLLM_LOG = LOG_DIR / 'vllm.log'
VLLM_URL = 'http://127.0.0.1:8000/v1/models'

def vllm_is_responsive():
    try:
        with urllib.request.urlopen(VLLM_URL, timeout=5) as resp:
            json.loads(resp.read().decode())
            return True
    except Exception:
        return False

if vllm_is_responsive():
    print('vLLM already running and responsive — skipping startup.')
    vllm_proc = None
else:
    vllm_bin = shutil.which('vllm')
    if not vllm_bin:
        raise RuntimeError('vllm binary not found. Run the pip install cell first.')

    cmd = [
        vllm_bin, 'serve', VLLM_MODEL_REPO,
        '--served-model-name', VLLM_SERVED_NAME,
        '--quantization', VLLM_QUANTIZATION,
        '--port', '8000',
        '--max-model-len', MAX_MODEL_LEN,
        '--gpu-memory-utilization', GPU_MEM_UTIL,
        '--max-num-seqs', MAX_NUM_SEQS,
        '--tensor-parallel-size', '1',
        '--tool-call-parser', 'gemma4',
        '--enable-auto-tool-choice',
        '--reasoning-parser', 'gemma4',
        '--mm-processor-kwargs', f'{{"max_soft_tokens": {MAX_SOFT_TOKENS}}}',
        '--limit-mm-per-prompt', '{"image": 4, "audio": 5}',
        '--async-scheduling',
    ] + VLLM_EXTRA_ARGS.split()

    print('vLLM command:', ' '.join(cmd))

    log_file = VLLM_LOG.open('w', encoding='utf-8')
    vllm_proc = subprocess.Popen(
        cmd,
        cwd=str(REPO_DIR),
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )

    print(f'\nStarted vLLM pid={vllm_proc.pid} — streaming output until ready...')
    print('(Safe to interrupt — vLLM will keep running in the background)\n')

    deadline = time.time() + 900
    try:
        for line in vllm_proc.stdout:
            log_file.write(line)
            log_file.flush()
            print(line, end='', flush=True)
            if vllm_proc.poll() is not None:
                break
            if 'Application startup complete' in line:
                break
            if time.time() > deadline:
                raise TimeoutError('Timed out (900s) waiting for vLLM.')
    except KeyboardInterrupt:
        print('\nCell interrupted — vLLM is still running in the background.')

    log_file.close()

    if vllm_proc.poll() is not None:
        tail = VLLM_LOG.read_text(errors='ignore').splitlines()[-40:]
        raise RuntimeError('vLLM exited early:\n' + '\n'.join(tail))

    if not vllm_is_responsive():
        for _ in range(10):
            time.sleep(1)
            if vllm_is_responsive():
                break
        else:
            raise RuntimeError('vLLM printed startup complete but /v1/models is not responding.')

print('\nvLLM is ready.')

vLLM command: /usr/local/bin/vllm serve google/gemma-4-E4B-it --served-model-name gemma-4-e4b-it --quantization bitsandbytes --port 8000 --max-model-len 2048 --gpu-memory-utilization 0.75 --max-num-seqs 1 --tensor-parallel-size 1 --tool-call-parser gemma4 --enable-auto-tool-choice --reasoning-parser gemma4 --mm-processor-kwargs {"max_soft_tokens": 140} --limit-mm-per-prompt {"image": 4, "audio": 5} --async-scheduling --enforce-eager --max-num-batched-tokens 512

Started vLLM pid=40807 — streaming output until ready...
(Safe to interrupt — vLLM will keep running in the background)

(APIServer pid=40807) INFO 05-14 12:51:18 [utils.py:299] 
(APIServer pid=40807) INFO 05-14 12:51:18 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=40807) INFO 05-14 12:51:18 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.20.1
(APIServer pid=40807) INFO 05-14 12:51:18 [utils.py:299]   █▄█▀ █     █     █     █  model   google/gemma-4-E4B-it
(APIServer pid=40807) INFO 05-14 12:51:18 [utils.py:299

## Start `app_blind.py` In Notebook Mode

The app is bound to loopback and runs with `APP_DISABLE_SSL=1`. HTTPS is provided by the tunnel, not by uvicorn directly.

In [ ]:
import sys
APP_LOG = LOG_DIR / 'app_blind.log'

if 'app_proc' in globals() and app_proc.poll() is None:
    raise RuntimeError('app_blind.py appears to already be running in this notebook session.')

app_env = os.environ.copy()
app_env.update({
    'APP_HOST': APP_HOST,
    'APP_PORT': APP_PORT,
    'APP_DISABLE_SSL': '1',
    'VLLM_MODEL_ID': VLLM_SERVED_NAME,
    'SPATIALSENSE_TIPSV2_SHORT_SIDE': TIPS_SHORT_SIDE,
})

with APP_LOG.open('w', encoding='utf-8') as log_file:
    app_proc = subprocess.Popen(
        [sys.executable, 'app_blind.py'],
        cwd=str(REPO_DIR),
        env=app_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )

print(f'Started app_blind.py with pid={app_proc.pid}. Waiting for local app...')

deadline = time.time() + 300
local_url = f'http://127.0.0.1:{APP_PORT}/'
while time.time() < deadline:
    if app_proc.poll() is not None:
        raise RuntimeError('app_blind.py exited early. Inspect /tmp/gemma_demo_logs/app_blind.log')
    try:
        with urllib.request.urlopen(local_url, timeout=5) as resp:
            html = resp.read().decode(errors='ignore')
            print(f'Local app responded with {len(html)} bytes')
            break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError('Timed out waiting for app_blind.py to become ready.')

print(f'Local app is ready on {local_url}')

Started app_blind.py with pid=41961. Waiting for local app...
Local app responded with 40207 bytes
Local app is ready on http://127.0.0.1:7862/


In [11]:
print(open('/tmp/gemma_demo_logs/app_blind.log').read())


                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

## Download `cloudflared` And Open A Quick Tunnel

This creates the HTTPS URL you can open on your phone.

In [12]:
import stat

if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'amd64'):
    raise RuntimeError(f'Expected Linux x86_64 runtime. Found: {platform.system()} {platform.machine()}')

cloudflared_path = LOG_DIR / 'cloudflared'
if not cloudflared_path.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared_path,
    )
    cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IEXEC)

print(f'cloudflared ready at {cloudflared_path}')

cloudflared ready at /tmp/gemma_demo_logs/cloudflared


In [13]:
import re

TUNNEL_LOG = LOG_DIR / 'cloudflared.log'

if 'cloudflared_proc' in globals() and cloudflared_proc.poll() is None:
    raise RuntimeError('cloudflared already appears to be running in this notebook session.')

tunnel_target = f'http://127.0.0.1:{APP_PORT}'
tunnel_log_file = TUNNEL_LOG.open('w', encoding='utf-8')
cloudflared_proc = subprocess.Popen(
    [str(cloudflared_path), 'tunnel', '--url', tunnel_target],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 90
recent = []

while time.time() < deadline:
    line = cloudflared_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    tunnel_log_file.write(line)
    tunnel_log_file.flush()
    recent.append(line.rstrip())
    match = re.search(r'https://[-a-zA-Z0-9.]+trycloudflare.com', line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError('Did not receive a public tunnel URL. Recent output:\n' + '\n'.join(recent[-20:]))

print('Open this URL on your phone:')
print(public_url)

Open this URL on your phone:
https://interests-insulin-womens-rolled.trycloudflare.com


In [14]:
public_url

'https://interests-insulin-womens-rolled.trycloudflare.com'

In [15]:
print(open('/tmp/gemma_demo_logs/app_blind.log').read()[-3000:])

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [16]:
!tail -f /tmp/gemma_demo_logs/vllm.log


(APIServer pid=40807) INFO 05-14 12:54:46 [launcher.py:46] Route: /v1/messages/count_tokens, Methods: POST
(APIServer pid=40807) INFO 05-14 12:54:46 [launcher.py:46] Route: /inference/v1/generate, Methods: POST
(APIServer pid=40807) INFO 05-14 12:54:46 [launcher.py:46] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=40807) INFO 05-14 12:54:46 [launcher.py:46] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=40807) INFO 05-14 12:54:46 [launcher.py:46] Route: /generative_scoring, Methods: POST
(APIServer pid=40807) INFO 05-14 12:54:46 [launcher.py:46] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=40807) INFO 05-14 12:54:46 [launcher.py:46] Route: /v1/completions/render, Methods: POST
(APIServer pid=40807) INFO:     Started server process [40807]
(APIServer pid=40807) INFO:     Waiting for application startup.
(APIServer pid=40807) INFO:     Application startup complete.
^C


## Phone Test Checklist

1. Open the printed `https://...trycloudflare.com` URL on your phone.
2. Allow microphone permission when prompted.
3. Take a photo or use the blind-first flow.
4. Ask a spoken question.
5. Confirm you receive both text and audio response.

If something breaks, inspect the log helper cells below.

## Log Helpers

In [ ]:
def tail_file(path, lines=80):
    path = Path(path)
    if not path.exists():
        print(f'{path} does not exist')
        return
    content = path.read_text(encoding='utf-8', errors='ignore').splitlines()
    print('\n'.join(content[-lines:]))

print('tail_file(VLLM_LOG)')
print('tail_file(APP_LOG)')
print('tail_file(TUNNEL_LOG)')

## Cleanup

Run this when you are done with the demo session.

In [ ]:
for proc_name in ('cloudflared_proc', 'app_proc', 'vllm_proc'):
    proc = globals().get(proc_name)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=20)
        print(f'Stopped {proc_name}')